# 🦴 02 — Keypoint Extraction Demo

Visualize OpenPose keypoints and preprocessing steps.

In [ ]:
import sys
sys.path.insert(0, '..')
import os, json, glob, numpy as np, matplotlib.pyplot as plt
from data_pipeline.preprocessing import read_openpose_json, process_single_clip, normalize_keypoints, pad_or_truncate
KEYPOINTS_DIR = '../data/keypoints'

## 1. Load Raw Keypoints from a Clip

In [ ]:
clip_dirs = [os.path.join(KEYPOINTS_DIR, d) for d in os.listdir(KEYPOINTS_DIR)
             if os.path.isdir(os.path.join(KEYPOINTS_DIR, d))]
if clip_dirs:
    raw = process_single_clip(clip_dirs[0])
    print(f'Clip: {os.path.basename(clip_dirs[0])}')
    print(f'Shape: {raw.shape} (frames, features)')
else:
    raw = np.random.randn(80, 411).astype(np.float32)
    print('No clips found — using synthetic data')

## 2. Plot Body Skeleton

In [ ]:
def plot_skeleton(frame, ax, title=''):
    body = frame[:75].reshape(25, 3)
    x, y, c = body[:,0], body[:,1], body[:,2]
    connections = [(0,1),(1,2),(2,3),(3,4),(1,5),(5,6),(6,7),(1,8),(8,9),(9,10),(8,12),(12,13),(13,14)]
    for i,j in connections:
        if c[i]>0.1 and c[j]>0.1:
            ax.plot([x[i],x[j]], [y[i],y[j]], 'b-', lw=2, alpha=0.6)
    valid = c > 0.1
    ax.scatter(x[valid], y[valid], c='red', s=30, zorder=5)
    ax.invert_yaxis(); ax.set_aspect('equal'); ax.set_title(title); ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, idx in zip(axes, np.linspace(0, len(raw)-1, 4, dtype=int)):
    plot_skeleton(raw[idx], ax, f'Frame {idx}')
plt.suptitle('Body Skeleton at Different Frames', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 3. Temporal Movement of Wrists

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(raw[:, 12], color='#E91E63', lw=1.5)
axes[0].set_title('Right Wrist X over time'); axes[0].grid(True, alpha=0.3)
axes[1].plot(raw[:, 21], color='#4CAF50', lw=1.5)
axes[1].set_title('Left Wrist X over time'); axes[1].set_xlabel('Frame')
axes[1].grid(True, alpha=0.3)
plt.suptitle('Temporal Keypoint Movement', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Raw vs Normalized

In [ ]:
norm = normalize_keypoints(raw)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(raw.flatten(), bins=100, color='#FF9800', alpha=0.7)
axes[0].set_title(f'Raw (mean={raw.mean():.3f}, std={raw.std():.3f})')
axes[1].hist(norm.flatten(), bins=100, color='#4CAF50', alpha=0.7)
axes[1].set_title(f'Normalized (mean={norm.mean():.3f}, std={norm.std():.3f})')
plt.suptitle('Z-Score Normalization Effect', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Padding Effect

In [ ]:
padded, orig_len = pad_or_truncate(norm, 150)
print(f'Original: {raw.shape[0]} frames → Padded: {padded.shape[0]} frames')
fig, ax = plt.subplots(figsize=(14, 3))
energy = np.sum(padded**2, axis=1)
colors = ['#4CAF50' if i < orig_len else '#E0E0E0' for i in range(150)]
ax.bar(range(150), energy, color=colors, width=1.0)
ax.axvline(orig_len, color='red', ls='--', label=f'Original = {orig_len}')
ax.set_xlabel('Frame'); ax.set_ylabel('Energy'); ax.legend()
ax.set_title('Real (green) vs Padded (gray) Frames')
plt.tight_layout(); plt.show()
print(f'✅ Final shape: {padded.shape} — ready for model!')